# Sistem Penunjang Keputusan Seleksi Penerima Beasiswa Santri Berprestasi dan Kurang Mampu Menggunakan Metode TOPSIS

## Pondok Pesantren Kreatif Sukamakmur YBM PLN

Notebook ini menghitung rekomendasi prioritas beasiswa dari data pendaftar, disertai penjelasan,
rumus, tabel seluruh alternatif, dan contoh substitusi pada setiap tahap.
TOPSIS memilih alternatif yang dekat dengan solusi ideal positif dan jauh dari solusi ideal negatif.

**Cara memakai:** pilih kernel Python dari `.venv`, ubah **BOBOT** dan **ALTERNATIF_CONTOH**
di sel konfigurasi, lalu jalankan **Restart Kernel & Run All** agar semua tabel diperbarui.
Bobot awal: C1 30%, C2 25%, C3 20%, C4 15%, C5 10%.
Tidak perlu memasang pandas atau NumPy; perhitungan memakai pustaka standar Python.
IPython untuk tampilan sudah tersedia pada kernel Jupyter.

**Kesepakatan data:** C1 memakai **nominal Penghasilan (Rp)** dengan sifat **Cost**.
C2–C5 memakai **skor 1–5 dari CSV** dengan sifat **Benefit**. Perbedaan satuan ditangani
oleh normalisasi vektor. Skor C1 pada CSV ditampilkan sebagai referensi, tidak ikut dihitung.

**Batas penggunaan:** hasil berupa urutan rekomendasi untuk ditinjau panitia. Notebook tidak
menetapkan penerima karena kuota dan aturan kelayakan belum diberikan.


## 0. Persiapan tampilan dan pembacaan data

CSV dibaca dengan pemisah titik koma (`;`). Data kosong, ID ganda, angka tidak valid,
penghasilan negatif, atau skor di luar 1–5 akan menghentikan perhitungan dengan pesan yang jelas.
Seluruh nilai antara disimpan tanpa pembulatan; tabel menampilkan enam angka desimal.


In [1]:
from pathlib import Path
from html import escape
import csv
import hashlib
import json
import math
from IPython.display import HTML, Markdown, Math, FileLink, display

NAMA_CSV = "Data_Pendaftar_PSB_Skala_1-5.csv"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / NAMA_CSV).is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Folder data tidak ditemukan. Buka notebook dari folder proyek spk yang berisi data/.")
DATA_FILE = ROOT / "data" / NAMA_CSV

display(HTML("""<style>
.table-wrap {overflow-x:auto; max-width:100%; margin:16px 0 24px; background:#fff; color:#18342e;
border:1px solid #cedbd7; border-radius:8px;}
.table-wrap table {border-collapse:collapse; width:100%; font-size:13px; font-variant-numeric:tabular-nums;}
.table-wrap caption {text-align:left; padding:12px 14px; font-weight:600; color:#164e40; background:#eef6f2;}
.table-wrap th {background:#dfeee8; text-align:left; padding:10px 14px; white-space:nowrap;}
.table-wrap td {padding:9px 14px; border-top:1px solid #e3ebe7; white-space:nowrap; text-align:left;}
.table-wrap tr:nth-child(even) td {background:#f7faf9;}
.table-wrap:focus-visible {outline:3px solid #007c65; outline-offset:2px;}
.spk-note {border-left:4px solid #c58d22; padding:12px 16px; background:#fff8e6; color:#574118; margin:16px 0;}
</style>"""))
def load_data(path=DATA_FILE):
    """Baca CSV titik koma; tolak data hilang, duplikat, dan skor di luar 1–5."""
    path = Path(path)
    numeric = ["Penghasilan (Rp)", "Nilai Rata-Rata", "Jumlah Saudara",
               "C1 (Penghasilan)", "C2 (Prestasi)", "C3 (Tanggungan)", "C4 (Status Ortu)", "C5 (SKTM)"]
    text_cols = ["No", "Nama Lengkap Calon Santri", "Penghasilan Asli", "Status Orang Tua",
                 "Berkas SKTM", "Keterangan Konversi"]
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter=";")
        missing = set(numeric + text_cols) - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"Kolom wajib tidak ditemukan: {sorted(missing)}")
        rows, seen = [], set()
        for line, row in enumerate(reader, 2):
            if None in row or any(row.get(c) is None or not row[c].strip() for c in numeric + text_cols):
                raise ValueError(f"Baris {line}: data kosong atau jumlah kolom tidak sesuai.")
            row = {k: v.strip() for k, v in row.items()}
            if not row["No"].isdigit() or int(row["No"]) < 1:
                raise ValueError(f"Baris {line}: No harus bilangan bulat positif.")
            row["id"] = f"A{int(row['No'])}"
            if row["id"] in seen:
                raise ValueError(f"Baris {line}: ID {row['id']} duplikat.")
            seen.add(row["id"])
            for col in numeric:
                try:
                    value = float(row[col])
                except (ValueError, TypeError) as exc:
                    raise ValueError(f"Baris {line}: {col} harus numerik.") from exc
                if not math.isfinite(value) or value < 0:
                    raise ValueError(f"Baris {line}: {col} harus berhingga dan tidak negatif.")
                if col.startswith("C") and (value != int(value) or not 1 <= value <= 5):
                    raise ValueError(f"Baris {line}: {col} harus skor bulat 1–5.")
                if col == "Jumlah Saudara" and value != int(value):
                    raise ValueError(f"Baris {line}: Jumlah Saudara harus bilangan bulat.")
                if col == "Nilai Rata-Rata" and value > 100:
                    raise ValueError(f"Baris {line}: Nilai Rata-Rata harus 0–100.")
                row[col] = int(value) if value.is_integer() else value
            rows.append(row)
    if not rows:
        raise ValueError("CSV tidak berisi pendaftar.")
    return rows


def fmt(value, digits=6):
    if isinstance(value, float):
        return f"{value:,.{digits}f}".translate(str.maketrans({",": ".", ".": ","}))
    if isinstance(value, int):
        return f"{value:,}".replace(",", ".")
    return str(value)


def table(headers, rows, caption=""):
    head = "".join(f"<th scope='col'>{escape(str(h))}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(fmt(v))}</td>" for v in row) + "</tr>" for row in rows)
    return ("<div class='table-wrap' role='region' tabindex='0' aria-label='" + escape(caption, quote=True) +
            "'><table><caption>" + escape(caption) + "</caption><thead><tr>" + head +
            "</tr></thead><tbody>" + body + "</tbody></table></div>")


def matrix_table(rows, matrix, caption):
    return table(["Alternatif", "Nama santri"] + [c["code"] for c in CRITERIA],
                 [[row["id"], row["Nama Lengkap Calon Santri"], *values] for row, values in zip(rows, matrix)], caption)


def verification(row):
    return ("Verifikasi penghasilan 0 dari isian '-'" if row["Penghasilan Asli"] == "-"
            else "Tinjau asumsi nominal" if "Asumsi" in row["Keterangan Konversi"] else "Tinjau konversi sumber")


def tampil_tabel(headers, values, caption):
    display(HTML(table(headers, values, caption)))

def tampil_matriks(values, caption):
    display(HTML(matrix_table(data, values, caption)))

def rumus(latex, teks):
    # LaTeX untuk Jupyter dan versi teks untuk laporan HTML offline.
    display({"text/latex": "$$" + latex + "$$", "text/plain": teks}, raw=True)


## 1. Informasi kriteria, rubrik, dan bobot yang dapat diatur

**Cost** berarti nilai yang lebih kecil lebih diprioritaskan. **Benefit** berarti nilai
yang lebih besar lebih diprioritaskan. Sifat digunakan saat menentukan solusi ideal;
normalisasi vektor tidak perlu membalik nilai cost terlebih dahulu.

Ubah angka pada `BOBOT` di bawah. Angka boleh berbentuk proporsi (0.30) atau persentase
(30), asalkan konsisten. Sistem membagi setiap bobot dengan totalnya sehingga jumlah bobot efektif 1.
Bobot negatif, bukan angka, tidak berhingga, atau seluruhnya nol ditolak.


In [2]:
# ===== KONFIGURASI YANG DAPAT DIUBAH =====
BOBOT = {
    "C1": 0.30,
    "C2": 0.25,
    "C3": 0.20,
    "C4": 0.15,
    "C5": 0.10,
}
ALTERNATIF_CONTOH = "A1"  # Pilih A1 sampai A23 untuk contoh substitusi.

# Pemetaan sesuai kesepakatan: C1 nominal cost; C2–C5 skor benefit.
KRITERIA = [{'code': 'C1',
  'name': 'Penghasilan Orang Tua',
  'column': 'Penghasilan (Rp)',
  'type': 'cost',
  'unit': 'Rp/bulan',
  'reason': 'Indikator ekonomi riil paling krusial untuk sasaran zakat.'},
 {'code': 'C2',
  'name': 'Prestasi Akademik',
  'column': 'C2 (Prestasi)',
  'type': 'benefit',
  'unit': 'skor 1–5',
  'reason': 'Menjaga standar mutu bahwa penerima beasiswa memang berprestasi.'},
 {'code': 'C3',
  'name': 'Jumlah Tanggungan Ortu',
  'column': 'C3 (Tanggungan)',
  'type': 'benefit',
  'unit': 'skor 1–5',
  'reason': 'Mengukur seberapa berat beban pembagi ekonomi keluarga.'},
 {'code': 'C4',
  'name': 'Status Orang Tua',
  'column': 'C4 (Status Ortu)',
  'type': 'benefit',
  'unit': 'skor 1–5',
  'reason': 'Prioritas perlindungan bagi santri yatim, piatu, atau single parent.'},
 {'code': 'C5',
  'name': 'Keberadaan SKTM',
  'column': 'C5 (SKTM)',
  'type': 'benefit',
  'unit': 'skor 1–5',
  'reason': 'Validasi administratif dari pihak pemerintah/desa.'}]
CRITERIA = KRITERIA  # Digunakan fungsi tampilan matriks.
kode = [k["code"] for k in KRITERIA]
if set(BOBOT) != set(kode):
    raise ValueError("BOBOT harus memuat tepat C1, C2, C3, C4, dan C5.")
bobot_input = [BOBOT[k] for k in kode]
if any(isinstance(w, bool) or not isinstance(w, (int, float)) or not math.isfinite(w) or w < 0 for w in bobot_input):
    raise ValueError("Setiap bobot harus berupa angka berhingga dan tidak negatif.")
total_bobot = sum(bobot_input)
if not math.isfinite(total_bobot) or total_bobot <= 0:
    raise ValueError("Total bobot harus berhingga dan lebih besar dari nol.")
w = [nilai / total_bobot for nilai in bobot_input]
sifat = [k["type"] for k in KRITERIA]
if any(s not in {"cost", "benefit"} for s in sifat):
    raise ValueError("Sifat kriteria harus cost atau benefit.")
rumus(r"w_j = \frac{b_j}{\sum_{k=1}^{n} b_k}, \qquad \sum_{j=1}^{n}w_j=1",
      "w_j = b_j / Σ_k b_k; jumlah bobot efektif = 1")
tampil_tabel(
    ["Kode", "Kriteria", "Kolom yang dihitung", "Satuan", "Sifat", "Bobot input", "Bobot efektif", "Alasan"],
    [[k["code"], k["name"], k["column"], k["unit"], k["type"].title(), bobot_input[j], w[j], k["reason"]]
     for j, k in enumerate(KRITERIA)], "Informasi kriteria dan bobot efektif")
display(Markdown(f"Total bobot input = **{fmt(total_bobot)}**; total bobot efektif = **{fmt(sum(w))}**."))


w_j = b_j / Σ_k b_k; jumlah bobot efektif = 1

Kode,Kriteria,Kolom yang dihitung,Satuan,Sifat,Bobot input,Bobot efektif,Alasan
C1,Penghasilan Orang Tua,Penghasilan (Rp),Rp/bulan,Cost,"0,300000","0,300000",Indikator ekonomi riil paling krusial untuk sasaran zakat.
C2,Prestasi Akademik,C2 (Prestasi),skor 1–5,Benefit,"0,250000","0,250000",Menjaga standar mutu bahwa penerima beasiswa memang berprestasi.
C3,Jumlah Tanggungan Ortu,C3 (Tanggungan),skor 1–5,Benefit,"0,200000","0,200000",Mengukur seberapa berat beban pembagi ekonomi keluarga.
C4,Status Orang Tua,C4 (Status Ortu),skor 1–5,Benefit,"0,150000","0,150000","Prioritas perlindungan bagi santri yatim, piatu, atau single parent."
C5,Keberadaan SKTM,C5 (SKTM),skor 1–5,Benefit,"0,100000","0,100000",Validasi administratif dari pihak pemerintah/desa.


Total bobot input = **1,000000**; total bobot efektif = **1,000000**.

### Rubrik konversi dari gambar pengguna

| Kriteria | Skor 5 | Skor 4 | Skor 3 | Skor 2 | Skor 1 |
|---|---|---|---|---|---|
| C1 Penghasilan | < Rp1 juta | Rp1–2 juta | Rp2–3,5 juta | Rp3,5–5 juta | > Rp5 juta |
| C2 Prestasi | 90–100 | 80–89 | 70–79 | 60–69 | < 60 |
| C3 Tanggungan | > 4 | 4 | 3 | 2 | 1 |
| C4 Status orang tua | Yatim piatu | Yatim | Piatu | Cerai | Lengkap |
| C5 SKTM | Ada | — | — | — | Tidak ada |

**Mengapa skor C1 tidak dipakai?** Pada rubrik ini, penghasilan rendah sudah diberi skor tinggi.
Memperlakukan skor tersebut sebagai cost akan membalik prioritas. Agar C1 tetap cost,
matriks memakai nominal rupiah. Rubrik C1 di atas hanya referensi terhadap CSV.

Rentang C1 pada gambar berbagi titik batas (misalnya Rp2 juta). Rentang C2 juga tidak
menuliskan batas desimal secara eksplisit. Notebook mempertahankan konversi yang sudah
ada, tanpa membuat ulang ambang. Data A8 memiliki 0 saudara dan skor C3=1; kasus 0
belum tercantum pada rubrik, sehingga harus dikonfirmasi. Perubahan rubrik resmi perlu
diikuti pembaruan data sumber sebelum menghitung ulang.


In [3]:
data = load_data(DATA_FILE)
m, n = len(data), len(KRITERIA)
ids = [row["id"] for row in data]
if ALTERNATIF_CONTOH not in ids:
    raise ValueError(f"ALTERNATIF_CONTOH harus salah satu dari {', '.join(ids)}.")
i_contoh = ids.index(ALTERNATIF_CONTOH)
nama_contoh = data[i_contoh]["Nama Lengkap Calon Santri"]
display(Markdown(f"Data terbaca: **{m} calon santri**, **{n} kriteria**. Contoh sepanjang notebook: **{ALTERNATIF_CONTOH} — {nama_contoh}**."))
kolom_asli = [k for k in data[0] if k != "id"]
tampil_tabel(["Alternatif", *kolom_asli], [[r["id"], *[r[k] for k in kolom_asli]] for r in data],
             "Data sumber lengkap — seluruh pendaftar dan catatan konversi")


Data terbaca: **23 calon santri**, **5 kriteria**. Contoh sepanjang notebook: **A1 — MUHAMMAD HABIL ALI**.

Alternatif,No,Nama Lengkap Calon Santri,Penghasilan Asli,Penghasilan (Rp),Nilai Rata-Rata,Jumlah Saudara,Status Orang Tua,Berkas SKTM,C1 (Penghasilan),C2 (Prestasi),C3 (Tanggungan),C4 (Status Ortu),C5 (SKTM),Keterangan Konversi
A1,1,MUHAMMAD HABIL ALI,2.500.000,2.500.000,"88,500000",6,ORANG TUA LENGKAP,Ada,3,4,5,1,5,Sesuai nominal
A2,2,Muhammmad rafa,3 juta/bulan,3.000.000,"84,200000",4,ORANG TUA LENGKAP,Ada,3,4,4,1,5,Sesuai nominal
A3,3,RIFQI I'TISHAM,3.500.000,3.500.000,"86,750000",1,ORANG TUA LENGKAP,Ada,3,4,1,1,5,Sesuai nominal
A4,4,Muhammad Bakhir Alfitoni,Rp. 800.000/ bulan,800.000,85,4,ORANG TUA LENGKAP,Ada,5,4,4,1,5,Sesuai nominal
A5,5,Ahzam Abdul Majid,"1.700. 000,-",1.700.000,"87,400000",4,ORANG TUA LENGKAP,Ada,4,4,4,1,5,Sesuai nominal
A6,6,FIKRI AL HABIB,"RP.500.000,- / BULAN",500.000,"83,800000",2,YATIM,Ada,5,4,2,4,5,Sesuai nominal
A7,7,Saktiawan jordi,80.000 - 100.000/HARI,2.340.000,"82,500000",2,ORANG TUA LENGKAP,Ada,3,4,2,1,5,Harian Rp 90rb x 26 hari = Rp 2.340.000
A8,8,Ahmad sahrudin,Rp.1.500.000/Perbulan,1.500.000,"85,600000",0,YATIM,Ada,4,4,1,4,5,Sesuai nominal
A9,9,Muhamad Zaki Alauddin,1000000,1.000.000,"89,100000",2,YATIM,Ada,4,4,2,4,5,Sesuai nominal
A10,10,Zaahid Rizqi Muhammad,2.000.000 + 3.000.000,5.000.000,"86,300000",5,ORANG TUA LENGKAP,Ada,2,4,5,1,5,Gabungan Rp 2 jt + Rp 3 jt = Rp 5.000.000


### Pemeriksaan kualitas data sebelum perhitungan

Nilai nol hasil konversi isian `-` tetap mengikuti CSV, bukan diimputasi atau dihapus.
Karena C1 adalah cost, angka 0 dianggap penghasilan paling rendah dan memengaruhi hasil.
Jumlah saudara pada sumber digunakan sebagai proksi tanggungan melalui skor C3 yang tersedia.
Semua SKTM bernilai 5; kriteria yang konstan tetap ditampilkan tetapi tidak membedakan pendaftar.


In [4]:
catatan_data = []
for row in data:
    alasan = []
    if row["Penghasilan Asli"] == "-":
        alasan.append("Penghasilan '-' dikonversi menjadi Rp0 pada sumber; perlu verifikasi")
    elif row["Keterangan Konversi"] != "Sesuai nominal":
        alasan.append(row["Keterangan Konversi"])
    if row["Jumlah Saudara"] == 0:
        alasan.append("0 saudara diberi skor C3=1; kasus ini belum tercantum pada rubrik")
    if alasan:
        catatan_data.append([row["id"], row["Nama Lengkap Calon Santri"], "; ".join(alasan)])
tampil_tabel(["Alternatif", "Nama santri", "Catatan untuk panitia"], catatan_data,
             "Asumsi dan konversi data yang perlu ditinjau")
display(Markdown("**Untuk seluruh pendaftar:** verifikasi bahwa skor C3 dari jumlah saudara mewakili tanggungan orang tua yang dimaksud."))


Alternatif,Nama santri,Catatan untuk panitia
A7,Saktiawan jordi,Harian Rp 90rb x 26 hari = Rp 2.340.000
A8,Ahmad sahrudin,0 saudara diberi skor C3=1; kasus ini belum tercantum pada rubrik
A10,Zaahid Rizqi Muhammad,Gabungan Rp 2 jt + Rp 3 jt = Rp 5.000.000
A14,Rehan Purnama,Penghasilan '-' dikonversi menjadi Rp0 pada sumber; perlu verifikasi
A17,RAFLI SYAPUTRA,Rentang Rp 2-3 jt (rata-rata Rp 2.500.000)
A18,IMAM MUSTAQIM,Penghasilan '-' dikonversi menjadi Rp0 pada sumber; perlu verifikasi
A21,Ardiansyah,Asumsi < Rp 1.000.000
A22,Ahmad zidan fahrudin,Penghasilan '-' dikonversi menjadi Rp0 pada sumber; perlu verifikasi
A23,Haical Lukman Hakim,Penghasilan '-' dikonversi menjadi Rp0 pada sumber; perlu verifikasi


**Untuk seluruh pendaftar:** verifikasi bahwa skor C3 dari jumlah saudara mewakili tanggungan orang tua yang dimaksud.

## 2. Membentuk matriks keputusan X

Setiap baris adalah alternatif (calon santri) dan setiap kolom adalah kriteria.
Nilai xᵢⱼ menyatakan nilai alternatif ke-i pada kriteria ke-j.
C1 berasal dari nominal rupiah; C2–C5 berasal dari skor konversi CSV.
Pada tahap ini belum dilakukan pembobotan atau pemilihan nilai ideal.


In [5]:
X = [[row[k["column"]] for k in KRITERIA] for row in data]
rumus(r"X=[x_{ij}]_{m\times n},\quad i=1,\ldots,m;\quad j=1,\ldots,n",
      "X = [x_ij] berukuran m × n")
tampil_matriks(X, f"Matriks keputusan X — {m} alternatif × {n} kriteria")
tampil_tabel(["Kriteria", "Kolom sumber", "Nilai contoh", "Sifat"],
             [[k["code"], k["column"], X[i_contoh][j], sifat[j].title()] for j, k in enumerate(KRITERIA)],
             f"Pembentukan baris {ALTERNATIF_CONTOH} — {nama_contoh}")


X = [x_ij] berukuran m × n

Alternatif,Nama santri,C1,C2,C3,C4,C5
A1,MUHAMMAD HABIL ALI,2.500.000,4,5,1,5
A2,Muhammmad rafa,3.000.000,4,4,1,5
A3,RIFQI I'TISHAM,3.500.000,4,1,1,5
A4,Muhammad Bakhir Alfitoni,800.000,4,4,1,5
A5,Ahzam Abdul Majid,1.700.000,4,4,1,5
A6,FIKRI AL HABIB,500.000,4,2,4,5
A7,Saktiawan jordi,2.340.000,4,2,1,5
A8,Ahmad sahrudin,1.500.000,4,1,4,5
A9,Muhamad Zaki Alauddin,1.000.000,4,2,4,5
A10,Zaahid Rizqi Muhammad,5.000.000,4,5,1,5


Kriteria,Kolom sumber,Nilai contoh,Sifat
C1,Penghasilan (Rp),2.500.000,Cost
C2,C2 (Prestasi),4,Benefit
C3,C3 (Tanggungan),5,Benefit
C4,C4 (Status Ortu),1,Benefit
C5,C5 (SKTM),5,Benefit


## 3. Menormalisasi matriks keputusan R

Normalisasi vektor menyetarakan skala rupiah dan skor. Kuadratkan setiap nilai, jumlahkan
kuadrat per kolom, lalu ambil akar sebagai pembagi. Setiap xᵢⱼ dibagi pembagi kolomnya.
Indeks k pada penjumlahan berjalan melalui seluruh m alternatif.

Cost dan benefit menggunakan rumus normalisasi yang sama; arahnya dibedakan pada tahap ideal.
Jika suatu kolom seluruhnya nol, nilai normalisasinya ditetapkan 0 karena tidak memiliki pembeda.


In [6]:
X_kuadrat = [[nilai ** 2 for nilai in row] for row in X]
jumlah_kuadrat = [sum(row[j] for row in X_kuadrat) for j in range(n)]
pembagi = [math.sqrt(nilai) for nilai in jumlah_kuadrat]
R = [[nilai / pembagi[j] if pembagi[j] else 0.0 for j, nilai in enumerate(row)] for row in X]
rumus(r"s_j=\sum_{k=1}^{m}x_{kj}^{2},\quad d_j=\sqrt{s_j},\quad r_{ij}=\frac{x_{ij}}{d_j}",
      "s_j = Σ_k x_kj²; d_j = √s_j; r_ij = x_ij / d_j")
tampil_matriks(X_kuadrat, "Kuadrat setiap nilai keputusan — xᵢⱼ²")
tampil_tabel(["Kriteria", "Penjumlahan seluruh xᵢⱼ²", "Jumlah kuadrat sⱼ", "Pembagi √sⱼ"],
             [[kode[j], " + ".join(fmt(row[j]) for row in X_kuadrat), jumlah_kuadrat[j], pembagi[j]]
              for j in range(n)], "Perhitungan pembagi normalisasi per kriteria")
tampil_matriks(R, "Matriks ternormalisasi R — seluruh alternatif")
tampil_tabel(["Kriteria", "xᵢⱼ", "Pembagi", "Substitusi xᵢⱼ / pembagi", "rᵢⱼ"],
             [[kode[j], X[i_contoh][j], pembagi[j],
               f"{fmt(X[i_contoh][j])} / {fmt(pembagi[j])}" if pembagi[j] else "Kolom nol → 0",
               R[i_contoh][j]] for j in range(n)], f"Contoh normalisasi {ALTERNATIF_CONTOH}")


s_j = Σ_k x_kj²; d_j = √s_j; r_ij = x_ij / d_j

Alternatif,Nama santri,C1,C2,C3,C4,C5
A1,MUHAMMAD HABIL ALI,6.250.000.000.000,16,25,1,25
A2,Muhammmad rafa,9.000.000.000.000,16,16,1,25
A3,RIFQI I'TISHAM,12.250.000.000.000,16,1,1,25
A4,Muhammad Bakhir Alfitoni,640.000.000.000,16,16,1,25
A5,Ahzam Abdul Majid,2.890.000.000.000,16,16,1,25
A6,FIKRI AL HABIB,250.000.000.000,16,4,16,25
A7,Saktiawan jordi,5.475.600.000.000,16,4,1,25
A8,Ahmad sahrudin,2.250.000.000.000,16,1,16,25
A9,Muhamad Zaki Alauddin,1.000.000.000.000,16,4,16,25
A10,Zaahid Rizqi Muhammad,25.000.000.000.000,16,25,1,25


Kriteria,Penjumlahan seluruh xᵢⱼ²,Jumlah kuadrat sⱼ,Pembagi √sⱼ
C1,6.250.000.000.000 + 9.000.000.000.000 + 12.250.000.000.000 + 640.000.000.000 + 2.890.000.000.000 + 250.000.000.000 + 5.475.600.000.000 + 2.250.000.000.000 + 1.000.000.000.000 + 25.000.000.000.000 + 2.250.000.000.000 + 6.250.000.000.000 + 8.410.000.000.000 + 0 + 25.000.000.000.000 + 4.000.000.000.000 + 6.250.000.000.000 + 0 + 6.250.000.000.000 + 2.250.000.000.000 + 810.000.000.000 + 0 + 0,126.475.600.000.000,"11.246.137,114583"
C2,16 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 25 + 16 + 16 + 16 + 16 + 16 + 16 + 16 + 16,377,"19,416488"
C3,25 + 16 + 1 + 16 + 16 + 4 + 4 + 1 + 4 + 25 + 16 + 1 + 25 + 1 + 16 + 4 + 9 + 9 + 4 + 25 + 25 + 9 + 1,257,"16,031220"
C4,1 + 1 + 1 + 1 + 1 + 16 + 1 + 16 + 16 + 1 + 1 + 16 + 1 + 16 + 9 + 1 + 1 + 16 + 1 + 1 + 1 + 16 + 25,160,"12,649111"
C5,25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25 + 25,575,"23,979158"


Alternatif,Nama santri,C1,C2,C3,C4,C5
A1,MUHAMMAD HABIL ALI,"0,222299","0,206010","0,311891","0,079057","0,208514"
A2,Muhammmad rafa,"0,266758","0,206010","0,249513","0,079057","0,208514"
A3,RIFQI I'TISHAM,"0,311218","0,206010","0,062378","0,079057","0,208514"
A4,Muhammad Bakhir Alfitoni,"0,071136","0,206010","0,249513","0,079057","0,208514"
A5,Ahzam Abdul Majid,"0,151163","0,206010","0,249513","0,079057","0,208514"
A6,FIKRI AL HABIB,"0,044460","0,206010","0,124757","0,316228","0,208514"
A7,Saktiawan jordi,"0,208071","0,206010","0,124757","0,079057","0,208514"
A8,Ahmad sahrudin,"0,133379","0,206010","0,062378","0,316228","0,208514"
A9,Muhamad Zaki Alauddin,"0,088919","0,206010","0,124757","0,316228","0,208514"
A10,Zaahid Rizqi Muhammad,"0,444597","0,206010","0,311891","0,079057","0,208514"


Kriteria,xᵢⱼ,Pembagi,Substitusi xᵢⱼ / pembagi,rᵢⱼ
C1,2.500.000,"11.246.137,114583","2.500.000 / 11.246.137,114583","0,222299"
C2,4,"19,416488","4 / 19,416488","0,206010"
C3,5,"16,031220","5 / 16,031220","0,311891"
C4,1,"12,649111","1 / 12,649111","0,079057"
C5,5,"23,979158","5 / 23,979158","0,208514"


## 4. Menghitung matriks normalisasi terbobot Y

Kalikan setiap nilai normalisasi dengan bobot efektif kriterianya.
Bobot berlaku per kolom, sehingga semua alternatif pada kriteria yang sama memakai bobot yang sama.
Tabel bobot dan matriks R pada tahap sebelumnya menjadi input tahap ini.


In [7]:
Y = [[nilai * w[j] for j, nilai in enumerate(row)] for row in R]
rumus(r"y_{ij}=w_j\,r_{ij}", "y_ij = w_j × r_ij")
tampil_matriks(Y, "Matriks normalisasi terbobot Y — seluruh alternatif")
tampil_tabel(["Kriteria", "Bobot wⱼ", "rᵢⱼ", "Substitusi wⱼ × rᵢⱼ", "yᵢⱼ"],
             [[kode[j], w[j], R[i_contoh][j], f"{fmt(w[j])} × {fmt(R[i_contoh][j])}", Y[i_contoh][j]]
              for j in range(n)], f"Contoh pembobotan {ALTERNATIF_CONTOH}")


y_ij = w_j × r_ij

Alternatif,Nama santri,C1,C2,C3,C4,C5
A1,MUHAMMAD HABIL ALI,"0,066690","0,051503","0,062378","0,011859","0,020851"
A2,Muhammmad rafa,"0,080027","0,051503","0,049903","0,011859","0,020851"
A3,RIFQI I'TISHAM,"0,093365","0,051503","0,012476","0,011859","0,020851"
A4,Muhammad Bakhir Alfitoni,"0,021341","0,051503","0,049903","0,011859","0,020851"
A5,Ahzam Abdul Majid,"0,045349","0,051503","0,049903","0,011859","0,020851"
A6,FIKRI AL HABIB,"0,013338","0,051503","0,024951","0,047434","0,020851"
A7,Saktiawan jordi,"0,062421","0,051503","0,024951","0,011859","0,020851"
A8,Ahmad sahrudin,"0,040014","0,051503","0,012476","0,047434","0,020851"
A9,Muhamad Zaki Alauddin,"0,026676","0,051503","0,024951","0,047434","0,020851"
A10,Zaahid Rizqi Muhammad,"0,133379","0,051503","0,062378","0,011859","0,020851"


Kriteria,Bobot wⱼ,rᵢⱼ,Substitusi wⱼ × rᵢⱼ,yᵢⱼ
C1,"0,300000","0,222299","0,300000 × 0,222299","0,066690"
C2,"0,250000","0,206010","0,250000 × 0,206010","0,051503"
C3,"0,200000","0,311891","0,200000 × 0,311891","0,062378"
C4,"0,150000","0,079057","0,150000 × 0,079057","0,011859"
C5,"0,100000","0,208514","0,100000 × 0,208514","0,020851"


## 5. Menentukan solusi ideal positif A⁺ dan ideal negatif A⁻

Solusi ideal positif memuat nilai terbaik tiap kriteria; ideal negatif memuat nilai terburuk.
Untuk **C1 Cost**, nilai terbaik adalah minimum dan nilai terburuk maksimum.
Untuk **C2–C5 Benefit**, nilai terbaik adalah maksimum dan nilai terburuk minimum.
Vektor ideal merupakan titik pembanding matematis dan tidak harus dimiliki satu santri.


In [8]:
A_plus, A_minus, ringkasan_ideal = [], [], []
for j in range(n):
    kolom = [row[j] for row in Y]
    terkecil, terbesar = min(kolom), max(kolom)
    positif = terkecil if sifat[j] == "cost" else terbesar
    negatif = terbesar if sifat[j] == "cost" else terkecil
    A_plus.append(positif)
    A_minus.append(negatif)
    ringkasan_ideal.append([kode[j], sifat[j].title(), terkecil, terbesar,
                           "min" if sifat[j] == "cost" else "max", positif,
                           "max" if sifat[j] == "cost" else "min", negatif])
rumus(r"a_j^+=\begin{cases}\max_i y_{ij},&\mathrm{benefit}\\\min_i y_{ij},&\mathrm{cost}\end{cases}",
      "a_j⁺ = max_i y_ij untuk Benefit; min_i y_ij untuk Cost")
rumus(r"a_j^-=\begin{cases}\min_i y_{ij},&\mathrm{benefit}\\\max_i y_{ij},&\mathrm{cost}\end{cases}",
      "a_j⁻ = min_i y_ij untuk Benefit; max_i y_ij untuk Cost")
tampil_tabel(["Kriteria", "Sifat", "min(Y)", "max(Y)", "Aturan A⁺", "A⁺", "Aturan A⁻", "A⁻"],
             ringkasan_ideal, "Pemilihan solusi ideal per kriteria")
tampil_tabel(["Vektor", *kode], [["A⁺", *A_plus], ["A⁻", *A_minus]], "Vektor ideal positif dan negatif")
konstan = [kode[j] for j in range(n) if max(row[j] for row in X) == min(row[j] for row in X)]
display(Markdown("Kriteria tanpa variasi dalam data: **" + (", ".join(konstan) or "tidak ada") +
                 "**. Untuk kriteria konstan, A⁺ = A⁻ dan kontribusi jaraknya 0."))


a_j⁺ = max_i y_ij untuk Benefit; min_i y_ij untuk Cost

a_j⁻ = min_i y_ij untuk Benefit; max_i y_ij untuk Cost

Kriteria,Sifat,min(Y),max(Y),Aturan A⁺,A⁺,Aturan A⁻,A⁻
C1,Cost,"0,000000","0,133379",min,"0,000000",max,"0,133379"
C2,Benefit,"0,051503","0,064378",max,"0,064378",min,"0,051503"
C3,Benefit,"0,012476","0,062378",max,"0,062378",min,"0,012476"
C4,Benefit,"0,011859","0,059293",max,"0,059293",min,"0,011859"
C5,Benefit,"0,020851","0,020851",max,"0,020851",min,"0,020851"


Vektor,C1,C2,C3,C4,C5
A⁺,"0,000000","0,064378","0,062378","0,059293","0,020851"
A⁻,"0,133379","0,051503","0,012476","0,011859","0,020851"


Kriteria tanpa variasi dalam data: **C5**. Untuk kriteria konstan, A⁺ = A⁻ dan kontribusi jaraknya 0.

## 6. Menghitung jarak terhadap solusi ideal

Untuk setiap alternatif, kurangi nilai terbobot dengan masing-masing komponen ideal,
kuadratkan selisihnya, jumlahkan semua kriteria, kemudian ambil akar (jarak Euclidean).
Dᵢ⁺ yang lebih kecil menunjukkan kedekatan dengan ideal positif.
Dᵢ⁻ yang lebih besar menunjukkan jarak yang lebih jauh dari ideal negatif.


In [9]:
selisih_plus_kuadrat = [[(nilai - A_plus[j]) ** 2 for j, nilai in enumerate(row)] for row in Y]
selisih_minus_kuadrat = [[(nilai - A_minus[j]) ** 2 for j, nilai in enumerate(row)] for row in Y]
jumlah_plus = [sum(row) for row in selisih_plus_kuadrat]
jumlah_minus = [sum(row) for row in selisih_minus_kuadrat]
D_plus = [math.sqrt(nilai) for nilai in jumlah_plus]
D_minus = [math.sqrt(nilai) for nilai in jumlah_minus]
rumus(r"D_i^+=\sqrt{\sum_{j=1}^{n}(y_{ij}-a_j^+)^2},\quad D_i^-=\sqrt{\sum_{j=1}^{n}(y_{ij}-a_j^-)^2}",
      "D_i⁺ = √[Σ_j (y_ij − a_j⁺)²]; D_i⁻ = √[Σ_j (y_ij − a_j⁻)²]")
tampil_matriks(selisih_plus_kuadrat, "Kontribusi kuadrat jarak ke A⁺ — seluruh alternatif")
tampil_matriks(selisih_minus_kuadrat, "Kontribusi kuadrat jarak ke A⁻ — seluruh alternatif")
tampil_tabel(["Alternatif", "Nama santri", "Σ(yᵢⱼ−aⱼ⁺)²", "Dᵢ⁺", "Σ(yᵢⱼ−aⱼ⁻)²", "Dᵢ⁻"],
             [[r["id"], r["Nama Lengkap Calon Santri"], jumlah_plus[i], D_plus[i], jumlah_minus[i], D_minus[i]]
              for i, r in enumerate(data)], "Jumlah kontribusi dan hasil akar jarak")
tampil_tabel(["Kriteria", "yᵢⱼ", "aⱼ⁺", "(yᵢⱼ−aⱼ⁺)²", "aⱼ⁻", "(yᵢⱼ−aⱼ⁻)²"],
             [[kode[j], Y[i_contoh][j], A_plus[j], selisih_plus_kuadrat[i_contoh][j], A_minus[j],
               selisih_minus_kuadrat[i_contoh][j]] for j in range(n)],
             f"Rincian jarak per kriteria untuk {ALTERNATIF_CONTOH}")
display(Markdown(
    f"Contoh **{ALTERNATIF_CONTOH}**: D⁺ = √({fmt(jumlah_plus[i_contoh])}) = **{fmt(D_plus[i_contoh])}**; "
    f"D⁻ = √({fmt(jumlah_minus[i_contoh])}) = **{fmt(D_minus[i_contoh])}**."))


D_i⁺ = √[Σ_j (y_ij − a_j⁺)²]; D_i⁻ = √[Σ_j (y_ij − a_j⁻)²]

Alternatif,Nama santri,C1,C2,C3,C4,C5
A1,MUHAMMAD HABIL ALI,"0,004447","0,000166","0,000000","0,002250","0,000000"
A2,Muhammmad rafa,"0,006404","0,000166","0,000156","0,002250","0,000000"
A3,RIFQI I'TISHAM,"0,008717","0,000166","0,002490","0,002250","0,000000"
A4,Muhammad Bakhir Alfitoni,"0,000455","0,000166","0,000156","0,002250","0,000000"
A5,Ahzam Abdul Majid,"0,002057","0,000166","0,000156","0,002250","0,000000"
A6,FIKRI AL HABIB,"0,000178","0,000166","0,001401","0,000141","0,000000"
A7,Saktiawan jordi,"0,003896","0,000166","0,001401","0,002250","0,000000"
A8,Ahmad sahrudin,"0,001601","0,000166","0,002490","0,000141","0,000000"
A9,Muhamad Zaki Alauddin,"0,000712","0,000166","0,001401","0,000141","0,000000"
A10,Zaahid Rizqi Muhammad,"0,017790","0,000166","0,000000","0,002250","0,000000"


Alternatif,Nama santri,C1,C2,C3,C4,C5
A1,MUHAMMAD HABIL ALI,"0,004447","0,000000","0,002490","0,000000","0,000000"
A2,Muhammmad rafa,"0,002846","0,000000","0,001401","0,000000","0,000000"
A3,RIFQI I'TISHAM,"0,001601","0,000000","0,000000","0,000000","0,000000"
A4,Muhammad Bakhir Alfitoni,"0,012553","0,000000","0,001401","0,000000","0,000000"
A5,Ahzam Abdul Majid,"0,007749","0,000000","0,001401","0,000000","0,000000"
A6,FIKRI AL HABIB,"0,014410","0,000000","0,000156","0,001266","0,000000"
A7,Saktiawan jordi,"0,005035","0,000000","0,000156","0,000000","0,000000"
A8,Ahmad sahrudin,"0,008717","0,000000","0,000000","0,001266","0,000000"
A9,Muhamad Zaki Alauddin,"0,011386","0,000000","0,000156","0,001266","0,000000"
A10,Zaahid Rizqi Muhammad,"0,000000","0,000000","0,002490","0,000000","0,000000"


Alternatif,Nama santri,Σ(yᵢⱼ−aⱼ⁺)²,Dᵢ⁺,Σ(yᵢⱼ−aⱼ⁻)²,Dᵢ⁻
A1,MUHAMMAD HABIL ALI,"0,006863","0,082845","0,006938","0,083293"
A2,Muhammmad rafa,"0,008976","0,094741","0,004247","0,065170"
A3,RIFQI I'TISHAM,"0,013623","0,116718","0,001601","0,040014"
A4,Muhammad Bakhir Alfitoni,"0,003027","0,055017","0,013953","0,118124"
A5,Ahzam Abdul Majid,"0,004628","0,068029","0,009150","0,095656"
A6,FIKRI AL HABIB,"0,001885","0,043418","0,015831","0,125822"
A7,Saktiawan jordi,"0,007713","0,087824","0,005191","0,072046"
A8,Ahmad sahrudin,"0,004398","0,066316","0,009983","0,099914"
A9,Muhamad Zaki Alauddin,"0,002419","0,049181","0,012807","0,113167"
A10,Zaahid Rizqi Muhammad,"0,020206","0,142147","0,002490","0,049903"


Kriteria,yᵢⱼ,aⱼ⁺,(yᵢⱼ−aⱼ⁺)²,aⱼ⁻,(yᵢⱼ−aⱼ⁻)²
C1,"0,066690","0,000000","0,004447","0,133379","0,004447"
C2,"0,051503","0,064378","0,000166","0,051503","0,000000"
C3,"0,062378","0,062378","0,000000","0,012476","0,002490"
C4,"0,011859","0,059293","0,002250","0,011859","0,000000"
C5,"0,020851","0,020851","0,000000","0,020851","0,000000"


Contoh **A1**: D⁺ = √(0,006863) = **0,082845**; D⁻ = √(0,006938) = **0,083293**.

## 7. Menghitung nilai preferensi V

Nilai preferensi membandingkan jarak terhadap ideal negatif dengan total kedua jarak.
Semakin besar Vᵢ, semakin tinggi prioritas rekomendasi. Nilainya berada dalam rentang 0–1,
tetapi **bukan probabilitas diterima**.

Jika Dᵢ⁺ + Dᵢ⁻ = 0, seluruh alternatif tidak dapat dibedakan pada kriteria aktif.
Rumus 0/0 tidak terdefinisi; notebook memakai **konvensi skor netral 0,5**, dengan peringkat seri.
Konvensi ini hanya penanganan kasus khusus, bukan rumus TOPSIS standar.


In [10]:
V = [minus / (plus + minus) if plus + minus > 0 else 0.5 for plus, minus in zip(D_plus, D_minus)]
rumus(r"V_i=\frac{D_i^-}{D_i^++D_i^-},\quad 0\leq V_i\leq1", "V_i = D_i⁻ / (D_i⁺ + D_i⁻), 0 ≤ V_i ≤ 1")
tampil_tabel(["Alternatif", "Nama santri", "Dᵢ⁺", "Dᵢ⁻", "Dᵢ⁺ + Dᵢ⁻", "Vᵢ"],
             [[r["id"], r["Nama Lengkap Calon Santri"], D_plus[i], D_minus[i], D_plus[i] + D_minus[i], V[i]]
              for i, r in enumerate(data)], "Nilai preferensi seluruh alternatif — urutan sumber")
if D_plus[i_contoh] + D_minus[i_contoh] > 0:
    display(Markdown(f"Contoh **{ALTERNATIF_CONTOH}**: V = {fmt(D_minus[i_contoh])} / "
                     f"({fmt(D_plus[i_contoh])} + {fmt(D_minus[i_contoh])}) = **{fmt(V[i_contoh])}**."))
else:
    display(Markdown(f"Contoh **{ALTERNATIF_CONTOH}**: D⁺ + D⁻ = 0, sehingga dipakai skor netral **0,5**."))


V_i = D_i⁻ / (D_i⁺ + D_i⁻), 0 ≤ V_i ≤ 1

Alternatif,Nama santri,Dᵢ⁺,Dᵢ⁻,Dᵢ⁺ + Dᵢ⁻,Vᵢ
A1,MUHAMMAD HABIL ALI,"0,082845","0,083293","0,166138","0,501349"
A2,Muhammmad rafa,"0,094741","0,065170","0,159911","0,407541"
A3,RIFQI I'TISHAM,"0,116718","0,040014","0,156732","0,255300"
A4,Muhammad Bakhir Alfitoni,"0,055017","0,118124","0,173141","0,682243"
A5,Ahzam Abdul Majid,"0,068029","0,095656","0,163685","0,584391"
A6,FIKRI AL HABIB,"0,043418","0,125822","0,169240","0,743455"
A7,Saktiawan jordi,"0,087824","0,072046","0,159870","0,450655"
A8,Ahmad sahrudin,"0,066316","0,099914","0,166229","0,601059"
A9,Muhamad Zaki Alauddin,"0,049181","0,113167","0,162349","0,697064"
A10,Zaahid Rizqi Muhammad,"0,142147","0,049903","0,192050","0,259842"


Contoh **A1**: V = 0,083293 / (0,082845 + 0,083293) = **0,501349**.

## 8. Mengurutkan ranking rekomendasi

Urutkan nilai preferensi dari terbesar ke terkecil menggunakan nilai asli tanpa pembulatan.
Nilai yang berbeda paling banyak 10⁻¹² terhadap nilai acuan kelompok dianggap seri dan mendapat
peringkat sama. Peringkat mengikuti pola kompetisi (1, 1, 3); urutan tampilan anggota seri mengikuti
urutan sumber dan tidak menjadi penentu kelayakan.

Seluruh pendaftar ditampilkan. Tidak ada label diterima atau ditolak karena kuota belum ditentukan.


In [11]:
urutan = sorted(range(m), key=lambda i: -V[i])
peringkat = [0] * m
nilai_acuan, rank_acuan = None, None
for posisi, i in enumerate(urutan, 1):
    if nilai_acuan is None or abs(V[i] - nilai_acuan) > 1e-12:
        nilai_acuan, rank_acuan = V[i], posisi
    peringkat[i] = rank_acuan
rumus(r"V_{(1)}\geq V_{(2)}\geq\cdots\geq V_{(m)}", "Urutan rekomendasi: V_(1) ≥ V_(2) ≥ … ≥ V_(m)")
hasil_ranking = [[peringkat[i], data[i]["id"], data[i]["Nama Lengkap Calon Santri"],
                  D_plus[i], D_minus[i], V[i], verification(data[i])] for i in urutan]
tampil_tabel(["Peringkat", "Alternatif", "Nama santri", "D⁺", "D⁻", "V", "Catatan verifikasi"],
             hasil_ranking, "Hasil ranking rekomendasi beasiswa")
teratas = [i for i in urutan if peringkat[i] == 1]
daftar_teratas = "; ".join(data[i]["id"] + " — " + data[i]["Nama Lengkap Calon Santri"] for i in teratas)
display(Markdown(f"Peringkat pertama: **{daftar_teratas}**, dengan nilai preferensi **{fmt(V[urutan[0]])}**. "
                 "Hasil mengikuti bobot aktif pada notebook dan masih memerlukan verifikasi data oleh panitia."))


Urutan rekomendasi: V_(1) ≥ V_(2) ≥ … ≥ V_(m)

Peringkat,Alternatif,Nama santri,D⁺,D⁻,V,Catatan verifikasi
1,A18,IMAM MUSTAQIM,"0,030479","0,140279","0,821507",Verifikasi penghasilan 0 dari isian '-'
1,A22,Ahmad zidan fahrudin,"0,030479","0,140279","0,821507",Verifikasi penghasilan 0 dari isian '-'
3,A6,FIKRI AL HABIB,"0,043418","0,125822","0,743455",Tinjau konversi sumber
4,A23,Haical Lukman Hakim,"0,051537","0,141563","0,733107",Verifikasi penghasilan 0 dari isian '-'
5,A14,Rehan Purnama,"0,052884","0,138042","0,723015",Verifikasi penghasilan 0 dari isian '-'
6,A9,Muhamad Zaki Alauddin,"0,049181","0,113167","0,697064",Tinjau konversi sumber
7,A21,Ardiansyah,"0,054701","0,120218","0,687278",Tinjau asumsi nominal
8,A4,Muhammad Bakhir Alfitoni,"0,055017","0,118124","0,682243",Tinjau konversi sumber
9,A20,MOHAMMAD FAQIH MUBAROK,"0,063379","0,105865","0,625517",Tinjau konversi sumber
10,A11,MHD AlI SYAHBANA,"0,064595","0,100588","0,608948",Tinjau konversi sumber


Peringkat pertama: **A18 — IMAM MUSTAQIM; A22 — Ahmad zidan fahrudin**, dengan nilai preferensi **0,821507**. Hasil mengikuti bobot aktif pada notebook dan masih memerlukan verifikasi data oleh panitia.

## 9. Pemeriksaan konsistensi hasil

Pemeriksaan berikut memastikan jumlah bobot 1, normalisasi tiap kolom tidak nol memiliki
jumlah kuadrat 1, ideal cost/benefit dipilih pada arah yang benar, nilai preferensi berada pada
0–1, dan urutan rekomendasi menurun. Pemeriksaan dijalankan ulang setelah bobot diubah.


In [12]:
assert math.isclose(sum(w), 1.0, abs_tol=1e-12)
assert all(len(row) == n for row in X)
for j in range(n):
    if pembagi[j]:
        assert math.isclose(sum(row[j] ** 2 for row in R), 1.0, abs_tol=1e-12)
    nilai_kolom = [row[j] for row in Y]
    assert A_plus[j] == (min(nilai_kolom) if sifat[j] == "cost" else max(nilai_kolom))
    assert A_minus[j] == (max(nilai_kolom) if sifat[j] == "cost" else min(nilai_kolom))
assert all(math.isfinite(v) and 0 <= v <= 1 for v in V)
assert all(V[urutan[k]] >= V[urutan[k + 1]] for k in range(m - 1))
display(Markdown("**Pemeriksaan konsistensi lulus.** Perhitungan tidak membulatkan nilai sebelum ranking."))


**Pemeriksaan konsistensi lulus.** Perhitungan tidak membulatkan nilai sebelum ranking.

## 10. Ekspor hasil perhitungan

Sel berikut menyimpan data sumber yang dipakai, konfigurasi bobot, matriks keputusan,
kuadrat keputusan, normalisasi, normalisasi terbobot, solusi ideal, kontribusi jarak,
jarak dan preferensi, serta ranking ke folder `reports/`.
CSV menggunakan pemisah titik koma; JSON menyimpan nilai numerik tanpa pembulatan tampilan.
Menjalankan ulang sel ini akan memperbarui berkas hasil.


In [13]:
OUTPUT = ROOT / "reports"
OUTPUT.mkdir(exist_ok=True)

def simpan_csv(nama, headers, values):
    path = OUTPUT / nama
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(headers)
        writer.writerows(values)
    return path

berkas = []
berkas.append(simpan_csv("00_data_sumber.csv", ["Alternatif", *kolom_asli],
                         [[r["id"], *[r[k] for k in kolom_asli]] for r in data]))
berkas.append(simpan_csv("01_kriteria_bobot.csv", ["Kode", "Kriteria", "Kolom", "Sifat", "Bobot input", "Bobot efektif"],
                         [[k["code"], k["name"], k["column"], sifat[j], bobot_input[j], w[j]] for j, k in enumerate(KRITERIA)]))
for nama, matriks in [("02_matriks_keputusan.csv", X), ("03_kuadrat_keputusan.csv", X_kuadrat),
                       ("04_normalisasi.csv", R), ("05_normalisasi_terbobot.csv", Y),
                       ("07_kuadrat_jarak_positif.csv", selisih_plus_kuadrat),
                       ("08_kuadrat_jarak_negatif.csv", selisih_minus_kuadrat)]:
    berkas.append(simpan_csv(nama, ["Alternatif", "Nama santri", *kode],
                             [[r["id"], r["Nama Lengkap Calon Santri"], *matriks[i]] for i, r in enumerate(data)]))
berkas.append(simpan_csv("06_pembagi_dan_ideal.csv", ["Kriteria", "Jumlah kuadrat", "Pembagi", "A+", "A-"],
                         [[kode[j], jumlah_kuadrat[j], pembagi[j], A_plus[j], A_minus[j]] for j in range(n)]))
berkas.append(simpan_csv("09_jarak_preferensi.csv", ["Alternatif", "D+", "D-", "Total jarak", "V"],
                         [[data[i]["id"], D_plus[i], D_minus[i], D_plus[i] + D_minus[i], V[i]] for i in range(m)]))
berkas.append(simpan_csv("10_ranking.csv", ["Peringkat", "Alternatif", "Nama santri", "D+", "D-", "V", "Catatan verifikasi"], hasil_ranking))
hasil_lengkap = {
    "sumber": DATA_FILE.name, "sha256_sumber": hashlib.sha256(DATA_FILE.read_bytes()).hexdigest(),
    "bobot_input": BOBOT, "bobot_efektif": dict(zip(kode, w)), "kriteria": KRITERIA,
    "data_sumber": data, "X": X, "X_kuadrat": X_kuadrat, "jumlah_kuadrat": jumlah_kuadrat,
    "pembagi": pembagi, "R": R, "Y": Y, "A_plus": A_plus, "A_minus": A_minus,
    "kuadrat_jarak_plus": selisih_plus_kuadrat, "kuadrat_jarak_minus": selisih_minus_kuadrat,
    "D_plus": D_plus, "D_minus": D_minus, "V": V, "peringkat": peringkat, "urutan_indeks": urutan,
    "catatan_data": catatan_data,
}
json_path = OUTPUT / "hasil_lengkap.json"
json_path.write_text(json.dumps(hasil_lengkap, ensure_ascii=False, indent=2), encoding="utf-8")
berkas.append(json_path)
for path in berkas:
    display(FileLink(str(path.relative_to(ROOT))))
display(Markdown(f"**{len(berkas)} berkas hasil disimpan** di folder reports/. Ubah BOBOT lalu jalankan ulang seluruh notebook untuk simulasi lain."))


/Users/user/code/personal/kampus/spk/reports/00_data_sumber.csv

/Users/user/code/personal/kampus/spk/reports/01_kriteria_bobot.csv

/Users/user/code/personal/kampus/spk/reports/02_matriks_keputusan.csv

/Users/user/code/personal/kampus/spk/reports/03_kuadrat_keputusan.csv

/Users/user/code/personal/kampus/spk/reports/04_normalisasi.csv

/Users/user/code/personal/kampus/spk/reports/05_normalisasi_terbobot.csv

... (3 outputs left out)

/Users/user/code/personal/kampus/spk/reports/09_jarak_preferensi.csv

/Users/user/code/personal/kampus/spk/reports/10_ranking.csv

/Users/user/code/personal/kampus/spk/reports/hasil_lengkap.json

**12 berkas hasil disimpan** di folder reports/. Ubah BOBOT lalu jalankan ulang seluruh notebook untuk simulasi lain.

## Catatan interpretasi dan referensi metode

- Bobot awal mengikuti konfigurasi pengguna, bukan hasil estimasi statistik.
- C1 menggunakan nominal bulanan dari hasil konversi pada CSV. Perkiraan harian, rentang,
  dan isian `-` perlu pemeriksaan sumber oleh panitia.
- C3 memakai skor yang tersedia; jumlah saudara belum tentu sama dengan tanggungan orang tua.
- SKTM seluruh pendaftar sama sehingga tidak membedakan ranking pada data saat ini.
- Skor tinggi menunjukkan prioritas relatif terhadap pendaftar dalam data dan bobot aktif.
  Penambahan atau penghapusan pendaftar dapat mengubah normalisasi, ideal, dan ranking.
- Ranking tidak menggantikan verifikasi berkas, penilaian kelayakan, atau keputusan panitia.

Urutan normalisasi, pembobotan, ideal, jarak, dan preferensi mengikuti TOPSIS dengan normalisasi
vektor. Contoh uraian rumus pada penelitian primer:
[A robust multiple-objective decision-making paradigm based on the water–energy–food security nexus under changing climate uncertainties](https://pmc.ncbi.nlm.nih.gov/articles/PMC8536774/).
Rubrik penilaian mengikuti gambar yang diberikan pengguna; sumber data adalah CSV lokal.
